In [4]:
import numpy as np

# ─── SCORE PARSING ────────────────────────────────────────────────────────────

def parse_score(score_str):
    result = []
    for s in score_str.strip().split():
        p1g, p2g = map(int, s.split('-'))
        result.append((p1g, p2g))
    return result

def is_set_complete(p1g, p2g):
    if p1g == 7 and p2g == 6: return True
    if p2g == 7 and p1g == 6: return True
    if p1g >= 6 and p1g - p2g >= 2: return True
    if p2g >= 6 and p2g - p1g >= 2: return True
    return False

def parse_match_state(score_str, best_of):
    sets_won = [0, 0]
    current_set_games = None
    for p1g, p2g in parse_score(score_str):
        if is_set_complete(p1g, p2g):
            if p1g > p2g: sets_won[0] += 1
            else:         sets_won[1] += 1
        else:
            current_set_games = (p1g, p2g)
            break
    sets_needed = best_of // 2 + 1
    assert sets_won[0] < sets_needed and sets_won[1] < sets_needed, \
        "Match is already over according to the score string."
    return sets_won, current_set_games

_NOTATION = {'0': 0, '15': 1, '30': 2, '40': 3, 'Ad': 4, 'AD': 4, 'A': 4}

def parse_game_score(game_score_str, is_tiebreak=False):
    s = game_score_str.strip()
    if not s or s == "0-0":
        return (0, 0)
    left, right = s.split('-')
    if is_tiebreak:
        return (int(left), int(right))
    return (_NOTATION[left], _NOTATION[right])

# ─── POINT / GAME / TIEBREAK / SET / MATCH ────────────────────────────────────

def sim_point(p1_serving):
    server, opp = (P1, P2) if p1_serving else (P2, P1)
    p_win_1st = (server['win_first']  + (1 - opp['return_first']))  / 2
    p_win_2nd = (server['win_second'] + (1 - opp['return_second'])) / 2
    if np.random.random() < server['first_in']:
        server_won = np.random.random() < p_win_1st
    else:
        server_won = np.random.random() < p_win_2nd
    return server_won if p1_serving else not server_won

def sim_game(p1_serving, start_score=(0, 0)):
    score = list(start_score)
    while True:
        p1_won = sim_point(p1_serving)
        score[0 if p1_won else 1] += 1
        if score[0] >= 4 and score[0] - score[1] >= 2: return True
        if score[1] >= 4 and score[1] - score[0] >= 2: return False

def sim_tiebreak(p1_serves_first, start_score=(0, 0)):
    score = list(start_score)
    point_count = score[0] + score[1]
    while True:
        p1_serves = p1_serves_first if point_count == 0 else p1_serves_first == (point_count % 2 == 0)
        p1_won = sim_point(p1_serves)
        score[0 if p1_won else 1] += 1
        point_count += 1
        if score[0] >= 7 and score[0] - score[1] >= 2: return True
        if score[1] >= 7 and score[1] - score[0] >= 2: return False

def sim_set(p1_serving, start_games=(0, 0), first_game_score=(0, 0)):
    games = list(start_games)
    first_game = True
    while True:
        score = first_game_score if first_game else (0, 0)
        first_game = False
        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1_serving, score)
            games[0 if p1_won_tb else 1] += 1
            return (games[0] > games[1]), p1_serving
        p1_won_game = sim_game(p1_serving, score)
        games[0 if p1_won_game else 1] += 1
        p1_serving = not p1_serving
        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1_serving)
            games[0 if p1_won_tb else 1] += 1
            return (games[0] > games[1]), p1_serving
        if games[0] >= 6 and games[0] - games[1] >= 2: return True,  p1_serving
        if games[1] >= 6 and games[1] - games[0] >= 2: return False, p1_serving

def sim_match_from_state(sets_won, current_set_games, first_game_score, p1_serving, best_of):
    sets_needed = best_of // 2 + 1
    sets = list(sets_won)
    first_set_games = current_set_games if current_set_games is not None else (0, 0)
    p1_won_set, p1_serving = sim_set(p1_serving, first_set_games, first_game_score)
    sets[0 if p1_won_set else 1] += 1
    while sets[0] < sets_needed and sets[1] < sets_needed:
        p1_won_set, p1_serving = sim_set(p1_serving)
        sets[0 if p1_won_set else 1] += 1
    return sets[0] > sets[1]

print("Simulation engine loaded.")

Simulation engine loaded.


In [5]:
# ─── JSON PARSER ──────────────────────────────────────────────────────────────

def _ratio(field):
    d, div = field.get("Dividend"), field.get("Divisor")
    if div and div > 0:
        return d / div
    return field["Percent"] / 100

def _player_stats(sets_array):
    agg = next(s for s in sets_array if s["SetNumber"] == 0)
    svc = agg["Stats"]["ServiceStats"]
    ret = agg["Stats"]["ReturnStats"]
    return {
        "first_in":      _ratio(svc["FirstServe"]),
        "win_first":     _ratio(svc["FirstServePointsWon"]),
        "win_second":    _ratio(svc["SecondServePointsWon"]),
        "return_first":  _ratio(ret["FirstServeReturnPointsWon"]),
        "return_second": _ratio(ret["SecondServeReturnPointsWon"]),
    }

def extract_match_state(data):
    match = data["Match"]
    t1    = match["PlayerTeam1"]
    t2    = match["PlayerTeam2"]

    p1_name = f"{t1['PlayerFirstNameFull']} {t1['PlayerLastName']}"
    p2_name = f"{t2['PlayerFirstNameFull']} {t2['PlayerLastName']}"

    p1_stats = _player_stats(t1["Sets"])
    p2_stats = _player_stats(t2["Sets"])

    p1_map = {s["SetNumber"]: s["SetScore"] for s in t1["Sets"]
              if s["SetNumber"] > 0 and s["SetScore"] is not None}
    p2_map = {s["SetNumber"]: s["SetScore"] for s in t2["Sets"]
              if s["SetNumber"] > 0 and s["SetScore"] is not None}
    set_nums  = sorted(set(p1_map) | set(p2_map))
    score_str = " ".join(f"{int(p1_map.get(n,0))}-{int(p2_map.get(n,0))}" for n in set_nums)

    pt = match["PlayerTeam"]
    ot = match["OpponentTeam"]
    if pt["Player"]["PlayerId"] == t1["PlayerId"]:
        g1, g2 = pt["GameScore"], ot["GameScore"]
    else:
        g1, g2 = ot["GameScore"], pt["GameScore"]
    game_score_str = f"{g1}-{g2}"

    # LastServer = who served the last completed game.
    # If a game is in progress (score ≠ 0-0), LastServer is the current server.
    # If score is 0-0 (new game not yet started), service has just switched → other player serves.
    last_server      = (match.get("LastServer") or "").upper()
    p1_id            = t1["PlayerId"].upper()
    game_in_progress = not (g1 == "0" and g2 == "0")

    if last_server in (p1_id, t2["PlayerId"].upper()):
        last_server_is_p1 = (last_server == p1_id)
        p1_serves = last_server_is_p1 if game_in_progress else not last_server_is_p1
    else:
        p1_serves = (match["ServerTeam"] == 1)  # fallback if LastServer missing

    best_of = match["NumberOfSets"]

    return {
        "p1_name":        p1_name,
        "p2_name":        p2_name,
        "p1_stats":       p1_stats,
        "p2_stats":       p2_stats,
        "score_str":      score_str,
        "game_score_str": game_score_str,
        "p1_serves":      p1_serves,
        "best_of":        best_of,
    }

# ── Dry-run against local example (0-0 game, LastServer=PL56 → Cerundolo serves) ──
import json, pathlib

with open(pathlib.Path("../apt_live_match_example.json")) as f:
    example = json.load(f)

state = extract_match_state(example)
print(f"Players:    {state['p1_name']} vs {state['p2_name']}")
print(f"Score:      {state['score_str']}")
print(f"Game score: {state['game_score_str']}  |  P1 serves: {state['p1_serves']}  (expect True = Cerundolo)")
print(f"Best of:    {state['best_of']}")

Players:    Francisco Cerundolo vs Tommy Paul
Score:      6-7 4-3
Game score: 0-0  |  P1 serves: True  (expect True = Cerundolo)
Best of:    3


In [6]:
# ─── CONFIG ───────────────────────────────────────────────────────────────────
URLS = [
    "https://www.atptour.com/-/Hawkeye/MatchStats/2026/8994/MS024",
    "https://www.atptour.com/-/Hawkeye/MatchStats/2026/741/QS007",  # update to second match
]

HEADERS = {
    "Referer": "https://www.atptour.com/en/scores/current",
    "Accept":  "application/json, text/plain, */*",
}

POLL_INTERVAL = 20
N             = 5_000

print("Config loaded.")

Config loaded.


In [7]:
# ─── LIVE POLL LOOP ───────────────────────────────────────────────────────────
# Run this cell to start tracking. Interrupt the kernel (■) to stop.

import time
from curl_cffi import requests as cffi_requests
from IPython.display import clear_output

done = set()   # URLs that finished

while len(done) < len(URLS):
    blocks = []

    for url in URLS:
        if url in done:
            blocks.append(f"[done] {url}")
            continue

        try:
            resp = cffi_requests.get(url, headers=HEADERS, impersonate="chrome120", timeout=10)
            if not resp.ok:
                blocks.append(f"[{url}] HTTP {resp.status_code}")
                continue
            data = resp.json()
        except Exception as e:
            blocks.append(f"[{url}] error: {e}")
            continue

        status = data.get("Match", {}).get("MatchStatus")
        if status != "P":
            done.add(url)
            blocks.append(f"[done] {url}  (status: {status})")
            continue

        state = extract_match_state(data)
        P1, P2 = state["p1_stats"], state["p2_stats"]

        sets_won, current_set_games = parse_match_state(state["score_str"], state["best_of"])
        in_tiebreak      = (current_set_games == (6, 6))
        first_game_score = parse_game_score(state["game_score_str"], is_tiebreak=in_tiebreak)

        wins = sum(
            sim_match_from_state(sets_won, current_set_games, first_game_score,
                                 state["p1_serves"], state["best_of"])
            for _ in range(N)
        )
        p1_prob = wins / N
        p2_prob = 1 - p1_prob

        cur_g       = current_set_games or (0, 0)
        in_prog_str = f"{cur_g[0]}-{cur_g[1]}" + (" [TB]" if in_tiebreak else "")
        server_str  = state["p1_name"] if state["p1_serves"] else state["p2_name"]

        lines = [
            f"{state['p1_name']} vs {state['p2_name']}  |  Best of {state['best_of']}",
            f"Sets: {sets_won[0]}-{sets_won[1]}  |  Current set: {in_prog_str}  |  Game: {state['game_score_str']}  |  Serving: {server_str}",
            f"",
            f"  {state['p1_name']:25s}  {p1_prob:.4f}",
            f"  {state['p2_name']:25s}  {p2_prob:.4f}",
        ]
        blocks.append("\n".join(lines))

    clear_output(wait=True)
    print(f"\n{'─'*50}\n".join(blocks))
    print(f"\n[{time.strftime('%H:%M:%S')}]  n={N:,}  |  next poll in {POLL_INTERVAL}s  (interrupt to stop)")

    time.sleep(POLL_INTERVAL)

print("All matches finished.")

[done] https://www.atptour.com/-/Hawkeye/MatchStats/2026/8994/MS024  (status: F)
──────────────────────────────────────────────────
[done] https://www.atptour.com/-/Hawkeye/MatchStats/2026/741/QS007

[11:02:57]  n=5,000  |  next poll in 20s  (interrupt to stop)
All matches finished.
